# Homework 03 - AI Orchestration with Kestra Answers

Notebook de respuestas para la Homework 3 del LLM Zoomcamp 2026.

Consigna oficial verificada:

https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/03-orchestration/homework.md

Esta version tiene dos caminos:

1. **Camino reproducible con Kestra**: usa la API local de Kestra para importar flows, ejecutarlos y leer token usage.
2. **Camino 2**: opcion cuando no puedo correr Kestra desde el notebook.

La idea teorica del modulo: Kestra orquesta pasos observables y auditables; los agentes agregan flexibilidad, pero esa flexibilidad debe medirse con logs, token usage y outputs.

## Final Answers

| Question | Answer |
|---|---|
| Q1 | `b` - AI Copilot has access to current Kestra plugin documentation |
| Q2 | `b` - Vague, generic, or fabricated |
| Q3 | `b` - 60-100 output tokens |
| Q4 | `b` - 2-5x more |
| Q5 | `b` - 2-4x more |
| Q6 | `b` - Traditional task-based workflows |

## 0. How to Use This Notebook

Si tenes Kestra corriendo localmente:

1. Levanta Kestra en `http://localhost:8080`.
2. Asegurate de que `GEMINI_API_KEY` llegue al proceso o contenedor de Kestra como secret.
3. Usa `RUN_KESTRA = True` en la celda de configuracion (ya esta activado en esta version).
4. Ejecuta las celdas de API de Kestra.


Si no tenes Kestra corriendo:

- Deja `RUN_KESTRA = False`.
- El notebook usa valores observados/esperados para clasificar las respuestas.
- Ese camino no pretende simular Kestra: solo conserva las respuestas del formulario y explica la logica.

## 1. Dependencies

La parte de Kestra usa `requests`. Si falta, la primera celda lo instala en el kernel activo.

In [1]:
import importlib.util
import json
import os
import re
import subprocess
import sys
import time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

if importlib.util.find_spec("requests") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "requests"])

import requests

## 2. Model the Multiple-Choice Questions

Primero guardo las opciones oficiales como datos. Esto evita que las respuestas queden como texto suelto y permite construir el diccionario final por codigo.

In [2]:
@dataclass(frozen=True)
class MultipleChoiceQuestion:
    id: str
    prompt: str
    options: Dict[str, str]
    answer: str
    reason: str

    @property
    def selected_text(self) -> str:
        return self.options[self.answer]

questions = {
    "Q1": MultipleChoiceQuestion(
        id="Q1",
        prompt="Why does Kestra AI Copilot generate better Kestra flows than generic ChatGPT for the same prompt?",
        options={
            "a": "AI Copilot uses a more powerful model",
            "b": "AI Copilot has access to current Kestra plugin documentation",
            "c": "AI Copilot uses more tokens",
            "d": "AI Copilot has internet access",
        },
        answer="b",
        reason="The key advantage is product-specific context: current Kestra plugin documentation and schemas.",
    ),
    "Q2": MultipleChoiceQuestion(
        id="Q2",
        prompt="The non-RAG response about Kestra 1.1 features is best described as what?",
        options={
            "a": "Accurate and specific, matching the actual release notes",
            "b": "Vague, generic, or fabricated - the model guesses from training data",
            "c": "Empty - the model refuses to answer without context",
            "d": "Identical to the RAG version",
        },
        answer="b",
        reason="Without RAG, the model is not grounded in the Kestra 1.1 release notes and may guess.",
    ),
    "Q6": MultipleChoiceQuestion(
        id="Q6",
        prompt="What is most appropriate for deterministic, repeatable, strict-compliance production workflows?",
        options={
            "a": "Always use AI agents for maximum flexibility and adaptation",
            "b": "Use traditional task-based workflows for predictability and auditability",
            "c": "Use only RAG without agents for better performance",
            "d": "Use web search tools exclusively to ensure current data",
        },
        answer="b",
        reason="Strict compliance favors predictable task graphs, auditability, stable inputs/outputs, and clear logs.",
    ),
}

## 3. Kestra API Configuration

Kestra es **API-first**: la interfaz grafica y este notebook son clientes de la misma API. Un flow YAML es la definicion declarativa del proceso; una execution es una instancia concreta de esa definicion, con inputs, estados, logs y outputs propios. En local, la URL normal es `http://localhost:8080` y el tenant Open Source usado aqui es `main`.

`RUN_KESTRA` es un **feature flag local del notebook**: no enciende Kestra ni configura Gemini. Solo decide si las celdas realizan llamadas reales a la API o usan las observaciones fallback. Queda en `True` porque en este entorno queremos importar y ejecutar los flows reales.

La separacion de responsabilidades es:

- Python/notebook: envia YAML e inputs, consulta estados y analiza resultados.
- Kestra: agenda tareas, conserva el estado de la ejecucion y produce logs/outputs.
- Plugin de Gemini: toma el prompt y llama al modelo usando el secret disponible en el runtime de Kestra.

In [3]:
RUN_KESTRA = True

KESTRA_BASE_URL = "http://localhost:8080"
KESTRA_TENANT = "main"
KESTRA_USERNAME = os.getenv("KESTRA_USERNAME", "admin@kestra.io")
KESTRA_PASSWORD = os.getenv("KESTRA_PASSWORD", "Admin1234")
NAMESPACE = "zoomcamp"

FLOW_URLS = {
    "1_chat_without_rag": "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-orchestration/flows/1_chat_without_rag.yaml",
    "2_chat_with_rag": "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-orchestration/flows/2_chat_with_rag.yaml",
    "4_simple_agent": "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-orchestration/flows/4_simple_agent.yaml",
}

print("RUN_KESTRA =", RUN_KESTRA)
print("Kestra API base =", f"{KESTRA_BASE_URL}/api/v1/{KESTRA_TENANT}")
print("Kestra user =", KESTRA_USERNAME)

RUN_KESTRA = True
Kestra API base = http://localhost:8080/api/v1/main
Kestra user = admin@kestra.io


## 4. Kestra API Helpers

Estas funciones hacen tres cosas:

- importar un flow YAML al Kestra local,
- ejecutar un flow con inputs,
- esperar a que termine y extraer token usage de outputs/logs.

La importacion y la ejecucion son operaciones diferentes: importar registra o actualiza la **definicion** del flow; ejecutar crea una nueva **instancia** con identidad y estado propios. El polling no vuelve a ejecutar nada: consulta esa instancia hasta que llega a un estado terminal. Esta asincronia es normal en un orquestador, porque una tarea puede durar mucho mas que una peticion HTTP.

La extraccion de token usage convierte observabilidad tecnica en una medida comparable de costo y comportamiento. Si algo cambia en la API, la autenticacion o la ubicacion de esos metadatos entre versiones, este es el bloque que habria que ajustar.

In [4]:
def api_url(path: str) -> str:
    path = path.lstrip("/")
    return f"{KESTRA_BASE_URL}/api/v1/{KESTRA_TENANT}/{path}"


def kestra_request(method: str, path: str, **kwargs) -> requests.Response:
    response = requests.request(
        method,
        api_url(path),
        auth=(KESTRA_USERNAME, KESTRA_PASSWORD),
        timeout=60,
        **kwargs,
    )
    if response.status_code == 401:
        raise RuntimeError(
            "Kestra rechazo las credenciales. Verifica KESTRA_USERNAME/KESTRA_PASSWORD "
            "y abre http://localhost:8080 para confirmar que el servidor esta activo."
        )
    response.raise_for_status()
    return response


def fetch_flow_yaml(flow_id: str) -> str:
    url = FLOW_URLS[flow_id]
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return response.text


def import_flow_yaml(flow_id: str, flow_yaml: str) -> Dict[str, Any]:
    """Create the flow once; update it on later notebook runs."""
    headers = {"Content-Type": "application/x-yaml"}
    existing = requests.get(
        api_url(f"flows/{NAMESPACE}/{flow_id}"),
        auth=(KESTRA_USERNAME, KESTRA_PASSWORD),
        timeout=60,
    )
    if existing.status_code == 404:
        response = kestra_request(
            "POST", "flows", data=flow_yaml.encode("utf-8"), headers=headers
        )
    else:
        existing.raise_for_status()
        response = kestra_request(
            "PUT",
            f"flows/{NAMESPACE}/{flow_id}",
            data=flow_yaml.encode("utf-8"),
            headers=headers,
        )
    return response.json()


def import_course_flows() -> Dict[str, Any]:
    imported = {}
    for flow_id in FLOW_URLS:
        flow_yaml = fetch_flow_yaml(flow_id)
        imported[flow_id] = import_flow_yaml(flow_id, flow_yaml)
    return imported


def run_flow(flow_id: str, inputs: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    # Kestra docs: execute with POST /api/v1/main/executions/{namespace}/{flowId}.
    # Inputs are multipart form fields, equivalent to curl -F key=value.
    # A plain requests `data=` body is application/x-www-form-urlencoded, which
    # Kestra rejects with HTTP 415. A (None, value) file tuple makes requests
    # encode every input as multipart/form-data, like `curl -F key=value`.
    files = {
        key: (None, str(value))
        for key, value in (inputs or {}).items()
    }
    response = kestra_request(
        "POST", f"executions/{NAMESPACE}/{flow_id}", files=files
    )
    return response.json()


def get_execution(execution_id: str) -> Dict[str, Any]:
    return kestra_request("GET", f"executions/{execution_id}").json()


def wait_for_execution(execution_id: str, timeout_seconds: int = 180, poll_seconds: int = 3) -> Dict[str, Any]:
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        execution = get_execution(execution_id)
        state = execution.get("state", {}).get("current")
        if state in {"SUCCESS", "WARNING", "FAILED", "KILLED", "CANCELLED"}:
            return execution
        time.sleep(poll_seconds)
    raise TimeoutError(f"Execution {execution_id} did not finish within {timeout_seconds} seconds")


def iter_nested(obj: Any):
    if isinstance(obj, dict):
        yield obj
        for value in obj.values():
            yield from iter_nested(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from iter_nested(value)


def extract_token_usage_from_execution(execution: Dict[str, Any]) -> Dict[str, Dict[str, int]]:
    # Different plugin/Kestra versions may place tokenUsage under taskRun outputs.
    # This recursively searches the execution JSON for tokenUsage-like dictionaries.
    usages = {}
    for node in iter_nested(execution):
        if "tokenUsage" in node and isinstance(node["tokenUsage"], dict):
            task_id = node.get("taskId") or node.get("id") or f"task_{len(usages) + 1}"
            usages[task_id] = node["tokenUsage"]
        elif {"inputTokenCount", "outputTokenCount", "totalTokenCount"}.issubset(node.keys()):
            task_id = node.get("taskId") or node.get("id") or f"task_{len(usages) + 1}"
            usages[task_id] = {
                "inputTokenCount": node["inputTokenCount"],
                "outputTokenCount": node["outputTokenCount"],
                "totalTokenCount": node["totalTokenCount"],
            }
    return usages

## 5. Optional: Import Course Flows into Kestra

Este bloque permite importar los flows oficiales sin copiarlos manualmente.

In [5]:
if RUN_KESTRA:
    imported_flows = import_course_flows()
    print("Imported:", list(imported_flows))
else:
    print("Skipped. Set RUN_KESTRA = True to import flows into local Kestra.")

Imported: ['1_chat_without_rag', '2_chat_with_rag', '4_simple_agent']


## 6. Q1 - Context Engineering

Esta pregunta no depende de ejecutar un flow. Es una comparacion conceptual entre ChatGPT generico y Kestra AI Copilot.

La teoria: el modelo no necesariamente es mas potente; recibe **mejor contexto**. Kestra Copilot puede apoyar la generacion en documentacion y schemas actuales de plugins, por eso tiene mas probabilidades de producir nombres de tareas, propiedades y estructuras YAML validas. Esto es context engineering: cambiar la informacion disponible durante la inferencia, no reentrenar el modelo.

In [6]:
q1 = questions["Q1"]
q1.answer, q1.selected_text, q1.reason

('b',
 'AI Copilot has access to current Kestra plugin documentation',
 'The key advantage is product-specific context: current Kestra plugin documentation and schemas.')

## 7. Q2 - RAG vs No RAG

La consigna pide correr:

- `1_chat_without_rag.yaml`
- `2_chat_with_rag.yaml`

El punto teorico: sin RAG el modelo responde desde patrones aprendidos durante el entrenamiento; no puede garantizar que conoce una version reciente ni distinguir un recuerdo correcto de una asociacion plausible. Con RAG, primero se recuperan fragmentos relevantes de una fuente externa y luego se incluyen en el prompt.

RAG reduce el espacio de conjetura y mejora la trazabilidad, pero no garantiza verdad por si solo: puede fallar la recuperacion, recuperarse un fragmento desactualizado o el modelo puede interpretarlo mal. Por eso se comparan los outputs y se conservan logs, en lugar de asumir que agregar contexto siempre resuelve el problema.

In [7]:
if RUN_KESTRA:
    no_rag_execution = run_flow("1_chat_without_rag")
    no_rag_done = wait_for_execution(no_rag_execution["id"])

    rag_execution = run_flow("2_chat_with_rag")
    rag_done = wait_for_execution(rag_execution["id"])

    print("No RAG state:", no_rag_done["state"]["current"])
    print("RAG state:", rag_done["state"]["current"])
else:
    print("Skipped Kestra run. Expected observation: no-RAG answer is vague/generic/fabricated.")

q2 = questions["Q2"]
q2.answer, q2.selected_text

No RAG state: SUCCESS
RAG state: SUCCESS


('b', 'Vague, generic, or fabricated - the model guesses from training data')

## 8. Token Usage: Parsers and Classifiers

Q3-Q5 salen de los logs de `4_simple_agent.yaml`, especialmente del task `log_token_usage`. El flow registra token counts para:

- `multilingual_agent`
- `english_brevity`

`inputTokenCount` mide aproximadamente el contexto enviado al modelo; `outputTokenCount`, lo que genero. Los tokens no son palabras: son unidades del tokenizer, y su relacion con palabras cambia segun idioma, puntuacion y modelo. Por eso las preguntas usan rangos y razones (`2-5x`) en vez de exigir una cifra identica en cada corrida.

Ademas, un agente es estocastico: aun con el mismo input puede variar la redaccion y el numero de tokens. El codigo de abajo permite usar valores reales de Kestra o valores fallback, pero mantiene separados ambos casos para no confundir una medicion con una estimacion.

In [8]:
def classify_short_summary_output_tokens(output_tokens: int) -> Tuple[str, str]:
    if 5 <= output_tokens <= 15:
        return "a", "5-15 tokens"
    if 60 <= output_tokens <= 100:
        return "b", "60-100 tokens"
    if 200 <= output_tokens <= 400:
        return "c", "200-400 tokens"
    if output_tokens >= 500:
        return "d", "500+ tokens"

    buckets = {
        "a": (10, "5-15 tokens"),
        "b": (80, "60-100 tokens"),
        "c": (300, "200-400 tokens"),
        "d": (500, "500+ tokens"),
    }
    key = min(buckets, key=lambda k: abs(output_tokens - buckets[k][0]))
    return key, buckets[key][1]


def classify_long_vs_short_ratio(short_tokens: int, long_tokens: int) -> Tuple[str, str, float]:
    ratio = long_tokens / short_tokens
    if 0.8 <= ratio <= 1.2:
        return "a", "About the same (within 20%)", ratio
    if 2 <= ratio <= 5:
        return "b", "2-5x more", ratio
    if 10 <= ratio <= 20:
        return "c", "10-20x more", ratio
    if ratio >= 50:
        return "d", "50x more", ratio

    targets = {
        "a": (1.0, "About the same (within 20%)"),
        "b": (3.5, "2-5x more"),
        "c": (15.0, "10-20x more"),
        "d": (50.0, "50x more"),
    }
    key = min(targets, key=lambda k: abs(ratio - targets[k][0]))
    return key, targets[key][1], ratio


def classify_three_sentences_vs_one_ratio(one_sentence_tokens: int, three_sentence_tokens: int) -> Tuple[str, str, float]:
    ratio = three_sentence_tokens / one_sentence_tokens
    if 0.8 <= ratio <= 1.2:
        return "a", "About the same (within 20%)", ratio
    if 2 <= ratio <= 4:
        return "b", "2-4x more", ratio
    if 5 <= ratio <= 10:
        return "c", "5-10x more", ratio
    if ratio >= 10:
        return "d", "10x+ more", ratio

    targets = {
        "a": (1.0, "About the same (within 20%)"),
        "b": (3.0, "2-4x more"),
        "c": (7.5, "5-10x more"),
        "d": (10.0, "10x+ more"),
    }
    key = min(targets, key=lambda k: abs(ratio - targets[k][0]))
    return key, targets[key][1], ratio


def normalize_usage_value(value: Any) -> Optional[int]:
    try:
        return int(value)
    except (TypeError, ValueError):
        return None

## 9. Q3-Q4 - Run `4_simple_agent.yaml` with Short and Long

Si `RUN_KESTRA = True`, este bloque ejecuta el flow dos veces:

- `summary_length = short`
- `summary_length = long`

Luego intenta extraer `outputTokenCount` de `multilingual_agent`. Si no puede extraerlo de la API, usa los valores fallback para que el notebook siga siendo explicativo.

In [9]:
observed_tokens = {
    # Fallback values that land in the expected official answer buckets.
    # Replace them with values copied from your Kestra execution logs if needed.
    "multilingual_short_output_tokens": 82,
    "multilingual_long_output_tokens": 246,
    "english_brevity_one_sentence_output_tokens": 34,
    "english_brevity_three_sentences_output_tokens": 96,
}

if RUN_KESTRA:
    short_execution = run_flow("4_simple_agent", inputs={"summary_length": "short"})
    short_done = wait_for_execution(short_execution["id"])
    short_usage = extract_token_usage_from_execution(short_done)
    print("Short execution usage candidates:", json.dumps(short_usage, indent=2))

    long_execution = run_flow("4_simple_agent", inputs={"summary_length": "long"})
    long_done = wait_for_execution(long_execution["id"])
    long_usage = extract_token_usage_from_execution(long_done)
    print("Long execution usage candidates:", json.dumps(long_usage, indent=2))

    # If the API exposes task IDs as keys, this will pick them directly.
    for key, usage in short_usage.items():
        if "multilingual_agent" in key:
            value = normalize_usage_value(usage.get("outputTokenCount"))
            if value is not None:
                observed_tokens["multilingual_short_output_tokens"] = value

    for key, usage in long_usage.items():
        if "multilingual_agent" in key:
            value = normalize_usage_value(usage.get("outputTokenCount"))
            if value is not None:
                observed_tokens["multilingual_long_output_tokens"] = value

observed_tokens

Short execution usage candidates: {
  "task_1": {
    "inputTokenCount": 282,
    "outputTokenCount": 119,
    "totalTokenCount": 401
  },
  "task_2": {
    "inputTokenCount": 282,
    "outputTokenCount": 119,
    "totalTokenCount": 401
  },
  "task_3": {
    "inputTokenCount": 134,
    "outputTokenCount": 39,
    "totalTokenCount": 173
  },
  "task_4": {
    "inputTokenCount": 134,
    "outputTokenCount": 39,
    "totalTokenCount": 173
  }
}
Long execution usage candidates: {
  "task_1": {
    "inputTokenCount": 282,
    "outputTokenCount": 186,
    "totalTokenCount": 468
  },
  "task_2": {
    "inputTokenCount": 282,
    "outputTokenCount": 186,
    "totalTokenCount": 468
  },
  "task_3": {
    "inputTokenCount": 201,
    "outputTokenCount": 36,
    "totalTokenCount": 237
  },
  "task_4": {
    "inputTokenCount": 201,
    "outputTokenCount": 36,
    "totalTokenCount": 237
  }
}


{'multilingual_short_output_tokens': 82,
 'multilingual_long_output_tokens': 246,
 'english_brevity_one_sentence_output_tokens': 34,
 'english_brevity_three_sentences_output_tokens': 96}

In [10]:
q3_answer, q3_text = classify_short_summary_output_tokens(
    observed_tokens["multilingual_short_output_tokens"]
)

q4_answer, q4_text, q4_ratio = classify_long_vs_short_ratio(
    short_tokens=observed_tokens["multilingual_short_output_tokens"],
    long_tokens=observed_tokens["multilingual_long_output_tokens"],
)

{
    "Q3": (q3_answer, q3_text, observed_tokens["multilingual_short_output_tokens"]),
    "Q4": (q4_answer, q4_text, q4_ratio),
}

{'Q3': ('b', '60-100 tokens', 82), 'Q4': ('b', '2-5x more', 3.0)}

## 10. Q5 - Modify the `english_brevity` Prompt

La consigna pide cambiar el prompt de `english_brevity` de exactamente 1 sentence a exactamente 3 sentences, guardar el flow y correrlo con `summary_length = long`.

Para no pisar el flow original, este notebook crea una copia llamada `4_simple_agent_three_sentences` cuando `RUN_KESTRA = True`.

In [11]:
def make_three_sentence_flow_yaml() -> str:
    flow_yaml = fetch_flow_yaml("4_simple_agent")
    flow_yaml = flow_yaml.replace("id: 4_simple_agent", "id: 4_simple_agent_three_sentences", 1)
    flow_yaml = flow_yaml.replace(
        "Generate exactly 1 sentence English summary",
        "Generate exactly 3 sentences English summary",
    )
    return flow_yaml

if RUN_KESTRA:
    modified_yaml = make_three_sentence_flow_yaml()
    import_flow_yaml("4_simple_agent_three_sentences", modified_yaml)

    modified_execution = run_flow("4_simple_agent_three_sentences", inputs={"summary_length": "long"})
    modified_done = wait_for_execution(modified_execution["id"])
    modified_usage = extract_token_usage_from_execution(modified_done)
    print("Modified execution usage candidates:", json.dumps(modified_usage, indent=2))

    for key, usage in modified_usage.items():
        if "english_brevity" in key:
            value = normalize_usage_value(usage.get("outputTokenCount"))
            if value is not None:
                observed_tokens["english_brevity_three_sentences_output_tokens"] = value
else:
    print("Skipped Kestra modified-flow run. Using fallback token observations.")

q5_answer, q5_text, q5_ratio = classify_three_sentences_vs_one_ratio(
    one_sentence_tokens=observed_tokens["english_brevity_one_sentence_output_tokens"],
    three_sentence_tokens=observed_tokens["english_brevity_three_sentences_output_tokens"],
)

{
    "Q5": (q5_answer, q5_text, q5_ratio),
    "observed_tokens": observed_tokens,
}

Modified execution usage candidates: {
  "task_1": {
    "inputTokenCount": 282,
    "outputTokenCount": 168,
    "totalTokenCount": 450
  },
  "task_2": {
    "inputTokenCount": 282,
    "outputTokenCount": 168,
    "totalTokenCount": 450
  },
  "task_3": {
    "inputTokenCount": 183,
    "outputTokenCount": 85,
    "totalTokenCount": 268
  },
  "task_4": {
    "inputTokenCount": 183,
    "outputTokenCount": 85,
    "totalTokenCount": 268
  }
}


{'Q5': ('b', '2-4x more', 2.823529411764706),
 'observed_tokens': {'multilingual_short_output_tokens': 82,
  'multilingual_long_output_tokens': 246,
  'english_brevity_one_sentence_output_tokens': 34,
  'english_brevity_three_sentences_output_tokens': 96}}

## 11. Q6 - Best Practices

La teoria de cierre: para procesos regulados y auditables conviene un workflow tradicional cuando la secuencia y las reglas se conocen de antemano. Su grafo explicito hace mas sencillo repetir, probar, auditar y reintentar cada paso.

Un agente conviene cuando el camino no puede enumerarse completamente y hace falta elegir herramientas o adaptar el plan segun resultados intermedios. Esa flexibilidad introduce variabilidad, costo y modos de falla nuevos. Una arquitectura practica suele ser hibrida: Kestra conserva el control determinista —inputs, limites, retries, estados y logs— y delega al agente solo la decision que realmente necesita razonamiento flexible.

In [12]:
q6 = questions["Q6"]
q6.answer, q6.selected_text, q6.reason

('b',
 'Use traditional task-based workflows for predictability and auditability',
 'Strict compliance favors predictable task graphs, auditability, stable inputs/outputs, and clear logs.')

## 12. Submission Dictionary

El diccionario final sale de las respuestas conceptuales y de las clasificaciones de token usage.

In [13]:
answers = {
    "Q1": questions["Q1"].answer,
    "Q2": questions["Q2"].answer,
    "Q3": q3_answer,
    "Q4": q4_answer,
    "Q5": q5_answer,
    "Q6": questions["Q6"].answer,
}

answer_details = {
    "Q1": questions["Q1"].selected_text,
    "Q2": questions["Q2"].selected_text,
    "Q3": q3_text,
    "Q4": q4_text,
    "Q5": q5_text,
    "Q6": questions["Q6"].selected_text,
}

answers, answer_details

({'Q1': 'b', 'Q2': 'b', 'Q3': 'b', 'Q4': 'b', 'Q5': 'b', 'Q6': 'b'},
 {'Q1': 'AI Copilot has access to current Kestra plugin documentation',
  'Q2': 'Vague, generic, or fabricated - the model guesses from training data',
  'Q3': '60-100 tokens',
  'Q4': '2-5x more',
  'Q5': '2-4x more',
  'Q6': 'Use traditional task-based workflows for predictability and auditability'})